In [ ]:
import os
import random
import numpy as np
import torch
import torchaudio
import pandas as pd
from sklearn.model_selection import train_test_split

# Base directory you printed
root = "/content/drive/MyDrive/SONARA/poultry_dataset"

poultry_base = os.path.join(
    root,
    "Poultry Vocalization Signal Dataset for Early Disease Detection",  # note the space/nbsp
)
smartears_base = os.path.join(
    root,
    "SmartEars A Practical Framework for Poultry Respiratory Monitoring via Spectrogram-Based Audio Classification and AI-Assisted Labeling",
)

print("poultry_base:", poultry_base, "exists:", os.path.isdir(poultry_base))
print("smartears_base:", smartears_base, "exists:", os.path.isdir(smartears_base))

# Unified classes
target_classes = ["Normal", "Respiratory Distress", "Noise"]
class_to_idx = {c: i for i, c in enumerate(target_classes)}
print("Class mapping:", class_to_idx)

# Exact subfolder names you listed
poultry_mapping = {
    "Healthy": "Normal",
    "Unhealthy": "Respiratory Distress",
    "Noise": "Noise",
}

smartears_mapping = {
    "Healthy": "Normal",
    "Sick": "Respiratory Distress",
    "None": "Noise",
}

poultry_base: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection exists: True
smartears_base: /content/drive/MyDrive/SONARA/poultry_dataset/SmartEars A Practical Framework for Poultry Respiratory Monitoring via Spectrogram-Based Audio Classification and AI-Assisted Labeling exists: True
Class mapping: {'Normal': 0, 'Respiratory Distress': 1, 'Noise': 2}


In [ ]:
def debug_collect(base_dir, mapping_dict):
    print("\n=== debug_collect from:", base_dir, "===")
    for src_class, tgt_class in mapping_dict.items():
        src_dir = os.path.join(base_dir, src_class)
        print("Looking for:", src_dir)
        if not os.path.isdir(src_dir):
            print("  -> NOT FOUND")
            continue
        num_files = 0
        for fname in os.listdir(src_dir):
            if fname.lower().endswith((".wav", ".flac")):
                num_files += 1
        print("  wav/flac files in", src_dir, ":", num_files)

debug_collect(poultry_base, poultry_mapping)
debug_collect(smartears_base, smartears_mapping)


=== debug_collect from: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection ===
Looking for: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Healthy
  wav/flac files in /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Healthy : 139
Looking for: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Unhealthy
  wav/flac files in /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Unhealthy : 121
Looking for: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Noise
  wav/flac files in /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Noise : 86

=== debug_collect from: /content/dri

In [ ]:
target_sr = 16000
n_mels = 64
n_fft = int(0.025 * target_sr)
hop_length = int(0.010 * target_sr)
max_frames = 400

mel_spec = torchaudio.transforms.MelSpectrogram(
    sample_rate=target_sr,
    n_fft=n_fft,
    hop_length=hop_length,
    n_mels=n_mels,
    center=True,
    power=2.0,
)
to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
delta = torchaudio.transforms.ComputeDeltas()
delta2 = torchaudio.transforms.ComputeDeltas()

def load_audio(path):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    return wav

def extract_feats(path):
    wav = load_audio(path)
    with torch.no_grad():
        ms = mel_spec(wav)          # (1, n_mels, T)
        ms_db = to_db(ms)
        d1 = delta(ms_db)
        d2 = delta2(d1)
        feat = torch.cat([ms_db, d1, d2], dim=0)  # (3, n_mels, T)
    return feat

def pad_or_crop(feat):
    C, F, T = feat.shape
    if T < max_frames:
        feat = torch.nn.functional.pad(feat, (0, max_frames - T))
    elif T > max_frames:
        start = random.randint(0, T - max_frames)
        feat = feat[:, :, start:start + max_frames]
    return feat

def collect_dataset(base_dir, mapping_dict):
    X_list, y_list, meta_list = [], [], []

    print("\n=== collect_dataset from:", base_dir, "===")
    for src_class, tgt_class in mapping_dict.items():
        src_dir = os.path.join(base_dir, src_class)
        print("Looking for:", src_dir)
        if not os.path.isdir(src_dir):
            print("  -> NOT FOUND")
            continue

        label_idx = class_to_idx[tgt_class]
        num_files = 0

        for fname in os.listdir(src_dir):
            if not fname.lower().endswith((".wav", ".flac")):
                continue
            num_files += 1
            fpath = os.path.join(src_dir, fname)
            try:
                feat = extract_feats(fpath)
                feat = pad_or_crop(feat)
                X_list.append(feat.numpy().astype(np.float32))
                y_list.append(label_idx)
                meta_list.append((fpath, tgt_class))
            except Exception as e:
                print("  Error processing", fpath, ":", e)

        print("  processed files in", src_dir, ":", num_files)

    if len(X_list) == 0:
        print("No files collected from", base_dir)
        return (
            np.empty((0, 3, n_mels, max_frames), dtype=np.float32),
            np.array([], dtype=np.int64),
            pd.DataFrame(columns=["filepath", "target_class"]),
        )

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    meta = pd.DataFrame(meta_list, columns=["filepath", "target_class"])
    return X, y, meta

print("Collecting Poultry...")
X_poultry, y_poultry, meta_poultry = collect_dataset(poultry_base, poultry_mapping)
print("Poultry shapes:", X_poultry.shape, y_poultry.shape)

print("Collecting SmartEars...")
X_smartears, y_smartears, meta_smartears = collect_dataset(smartears_base, smartears_mapping)
print("SmartEars shapes:", X_smartears.shape, y_smartears.shape)


=== collect_dataset from: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection ===
Looking for: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Healthy
  processed files in /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Healthy : 139
Looking for: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Unhealthy
  processed files in /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Unhealthy : 121
Looking for: /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Noise
  processed files in /content/drive/MyDrive/SONARA/poultry_dataset/Poultry Vocalization Signal Dataset for Early Disease Detection/Noise : 86
Poultry shapes: (346, 3, 64, 400

In [ ]:
X_all = np.concatenate([X_poultry, X_smartears], axis=0)
y_all = np.concatenate([y_poultry, y_smartears], axis=0)
meta_all = pd.concat([meta_poultry, meta_smartears], ignore_index=True)

print("Combined:", X_all.shape, y_all.shape)
print("Class counts:", np.bincount(y_all))

X_train, X_temp, y_train, y_temp, meta_train, meta_temp = train_test_split(
    X_all, y_all, meta_all,
    test_size=0.3,
    stratify=y_all,
    random_state=SEED,
)

X_val, X_test, y_val, y_test, meta_val, meta_test = train_test_split(
    X_temp, y_temp, meta_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=SEED,
)

print("Train:", X_train.shape, np.bincount(y_train))
print("Val:  ", X_val.shape,   np.bincount(y_val))
print("Test: ", X_test.shape,  np.bincount(y_test))

out_path = "/content/drive/MyDrive/SONARA/sonara_logmel_deltas_combined.npz"
np.savez_compressed(
    out_path,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    X_test=X_test,
    y_test=y_test,
    class_names=np.array(target_classes),
)
print("Saved new dataset to:", out_path)

Combined: (6346, 3, 64, 400) (6346,)
Class counts: [2139 2121 2086]
Train: (4442, 3, 64, 400) [1497 1485 1460]
Val:   (952, 3, 64, 400) [321 318 313]
Test:  (952, 3, 64, 400) [321 318 313]
Saved new dataset to: /content/drive/MyDrive/SONARA/sonara_logmel_deltas_combined.npz
